In [ ]:
# ============================================
# SETUP & IMPORTS
# ============================================

import nibabel as nib
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
import json
from tqdm import tqdm
import matplotlib.pyplot as plt
from scipy import ndimage

print("✅ Imports successful!")

# Paths (CHANGE THESE after you download BraTS)
BRATS_ROOT = Path(r"C:\Pranav Aditya\MP01\BraTS2020\Training")
SURVIVAL_CSV = Path(r"C:\Pranav Aditya\MP01\BraTS2020\survival_info.csv")
OUTPUT_DIR = Path(r"C:\Pranav Aditya\MP01\BraTS_Processed")
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

print(f"\nBraTS Root: {BRATS_ROOT}")
print(f"Output Directory: {OUTPUT_DIR}")

## 📊 Step 1: Load Survival Metadata

BraTS provides a CSV file with patient information:
- Age
- Survival days
- Resection status (GTR/STR)
- Tumor grade (HGG/LGG)

In [ ]:
# ============================================
# LOAD SURVIVAL METADATA
# ============================================

# Check if file exists
if not SURVIVAL_CSV.exists():
    print(f"⚠️ Survival CSV not found at {SURVIVAL_CSV}")
    print("This file should be included with BraTS download.")
    survival_df = None
else:
    survival_df = pd.read_csv(SURVIVAL_CSV)
    print(f"✅ Loaded survival data for {len(survival_df)} patients")
    print(f"\nColumns: {list(survival_df.columns)}")
    print(f"\nFirst 5 rows:")
    display(survival_df.head())
    
    # Convert to dictionary for easy lookup
    survival_dict = survival_df.set_index('Brats20ID').to_dict('index')
    print(f"\n✅ Created lookup dictionary for {len(survival_dict)} patients")

## 🧠 Step 2: Tumor Location Detection

We'll identify which brain region contains the tumor using the segmentation mask and anatomical landmarks.

In [ ]:
# ============================================
# BRAIN REGION DETECTION
# ============================================

def get_brain_region(center_of_mass, volume_shape):
    """
    Determine brain region from tumor center of mass.
    
    BraTS brain dimensions are roughly 240x240x155 voxels (1mm³ each)
    Coordinate system: (x, y, z) where z is axial slice
    
    Args:
        center_of_mass: (x, y, z) coordinates of tumor center
        volume_shape: (height, width, depth) of brain volume
    
    Returns:
        str: Brain region name
    """
    x, y, z = center_of_mass
    h, w, d = volume_shape
    
    # Normalize to 0-1
    x_norm = x / w
    y_norm = y / h
    z_norm = z / d
    
    # Anterior-Posterior (front-back)
    if y_norm < 0.4:
        ap_region = "frontal"
    elif y_norm > 0.7:
        ap_region = "occipital"
    else:
        ap_region = "central"
    
    # Left-Right
    if x_norm < 0.45:
        lr_region = "left"
    elif x_norm > 0.55:
        lr_region = "right"
    else:
        lr_region = "midline"
    
    # Superior-Inferior (top-bottom)
    if z_norm < 0.3:
        si_region = "inferior"
    elif z_norm > 0.7:
        si_region = "superior"
    else:
        si_region = "middle"
    
    # Temporal lobe specific detection
    if 0.3 < y_norm < 0.6 and 0.3 < z_norm < 0.5:
        ap_region = "temporal"
    
    # Combine regions
    if lr_region == "midline":
        region = f"{ap_region}_{si_region}"
    else:
        region = f"{lr_region}_{ap_region}_{si_region}"
    
    return region


def analyze_tumor(seg_mask):
    """
    Extract tumor characteristics from segmentation mask.
    
    BraTS labels:
    - 0: Background
    - 1: Necrotic/Non-enhancing tumor core (NCR/NET)
    - 2: Peritumoral edema (ED)
    - 4: GD-enhancing tumor (ET)
    
    Returns:
        dict: Tumor metadata
    """
    # Calculate volumes (each voxel is 1mm³)
    necrotic_volume = np.sum(seg_mask == 1)
    edema_volume = np.sum(seg_mask == 2)
    enhancing_volume = np.sum(seg_mask == 4)
    total_volume = necrotic_volume + edema_volume + enhancing_volume
    
    # Center of mass (tumor location)
    tumor_binary = seg_mask > 0
    if np.sum(tumor_binary) > 0:
        center_of_mass = ndimage.center_of_mass(tumor_binary)
        brain_region = get_brain_region(center_of_mass, seg_mask.shape)
    else:
        center_of_mass = (0, 0, 0)
        brain_region = "unknown"
    
    # Bounding box (tumor extent)
    if np.sum(tumor_binary) > 0:
        coords = np.argwhere(tumor_binary)
        min_coords = coords.min(axis=0)
        max_coords = coords.max(axis=0)
        tumor_size = max_coords - min_coords  # in voxels (mm)
    else:
        tumor_size = np.array([0, 0, 0])
    
    return {
        'total_volume_mm3': int(total_volume),
        'necrotic_volume_mm3': int(necrotic_volume),
        'edema_volume_mm3': int(edema_volume),
        'enhancing_volume_mm3': int(enhancing_volume),
        'center_of_mass': [float(x) for x in center_of_mass],
        'brain_region': brain_region,
        'tumor_size_mm': [int(x) for x in tumor_size],
        'has_necrosis': necrotic_volume > 0,
        'has_edema': edema_volume > 0,
        'has_enhancement': enhancing_volume > 0,
    }

print("✅ Tumor analysis functions defined")

## 🖼️ Step 3: Extract 2D Slices from 3D Volumes

Each BraTS case is a 3D volume (240×240×155 voxels).
We'll extract axial slices (horizontal cuts through the brain) that contain tumor.

In [ ]:
# ============================================
# SLICE EXTRACTION
# ============================================

def extract_slices_with_tumor(volume, seg_mask, num_slices=10):
    """
    Extract axial slices that contain tumor tissue.
    
    Args:
        volume: 3D MRI volume (H, W, D)
        seg_mask: 3D segmentation mask
        num_slices: Number of slices to extract
    
    Returns:
        list: [(slice_image, slice_idx, tumor_present_ratio), ...]
    """
    # Find slices with tumor
    tumor_slices = []
    for z in range(seg_mask.shape[2]):
        tumor_pixels = np.sum(seg_mask[:, :, z] > 0)
        if tumor_pixels > 100:  # At least 100 pixels of tumor
            tumor_ratio = tumor_pixels / (seg_mask.shape[0] * seg_mask.shape[1])
            tumor_slices.append((z, tumor_ratio))
    
    # Sort by tumor content
    tumor_slices.sort(key=lambda x: x[1], reverse=True)
    
    # Select top slices
    selected_slices = []
    for z, tumor_ratio in tumor_slices[:num_slices]:
        slice_img = volume[:, :, z]
        
        # Normalize to 0-255
        slice_img = (slice_img - slice_img.min()) / (slice_img.max() - slice_img.min() + 1e-8)
        slice_img = (slice_img * 255).astype(np.uint8)
        
        selected_slices.append((slice_img, z, tumor_ratio))
    
    return selected_slices


def process_patient(patient_dir, output_dir, survival_dict=None, num_slices=10):
    """
    Process a single BraTS patient.
    
    Args:
        patient_dir: Path to patient folder (e.g., BraTS20_Training_001)
        output_dir: Where to save processed images
        survival_dict: Patient metadata dictionary
        num_slices: Number of slices to extract per patient
    """
    patient_id = patient_dir.name
    
    # File paths
    t1ce_path = patient_dir / f"{patient_id}_t1ce.nii.gz"
    seg_path = patient_dir / f"{patient_id}_seg.nii.gz"
    
    if not t1ce_path.exists() or not seg_path.exists():
        print(f"⚠️ Missing files for {patient_id}")
        return None
    
    # Load NIfTI files
    t1ce_nii = nib.load(str(t1ce_path))
    seg_nii = nib.load(str(seg_path))
    
    t1ce_data = t1ce_nii.get_fdata()
    seg_data = seg_nii.get_fdata()
    
    # Analyze tumor
    tumor_info = analyze_tumor(seg_data)
    
    # Get patient metadata from survival CSV
    if survival_dict and patient_id in survival_dict:
        patient_meta = survival_dict[patient_id]
        age = patient_meta.get('Age', None)
        survival = patient_meta.get('Survival_days', None)
        resection = patient_meta.get('Extent_of_Resection', 'Unknown')
        grade = patient_meta.get('Grade', 'Unknown')
    else:
        age = None
        survival = None
        resection = 'Unknown'
        grade = 'Unknown'
    
    # Extract slices
    slices = extract_slices_with_tumor(t1ce_data, seg_data, num_slices)
    
    # Save each slice
    patient_output_dir = output_dir / patient_id
    patient_output_dir.mkdir(exist_ok=True, parents=True)
    
    metadata_list = []
    for i, (slice_img, slice_idx, tumor_ratio) in enumerate(slices):
        # Save image
        img_filename = f"{patient_id}_slice{slice_idx:03d}.png"
        img_path = patient_output_dir / img_filename
        
        # Convert grayscale to RGB
        rgb_img = np.stack([slice_img] * 3, axis=-1)
        Image.fromarray(rgb_img).save(img_path)
        
        # Create metadata
        metadata = {
            'filename': img_filename,
            'patient_id': patient_id,
            'slice_index': int(slice_idx),
            'tumor_ratio': float(tumor_ratio),
            
            # Tumor characteristics
            'tumor_grade': grade,
            'brain_region': tumor_info['brain_region'],
            'tumor_volume_mm3': tumor_info['total_volume_mm3'],
            'has_necrosis': tumor_info['has_necrosis'],
            'has_edema': tumor_info['has_edema'],
            'has_enhancement': tumor_info['has_enhancement'],
            
            # Patient data
            'age': age,
            'survival_days': survival,
            'resection_status': resection,
        }
        metadata_list.append(metadata)
    
    # Save patient metadata JSON
    json_path = patient_output_dir / f"{patient_id}_metadata.json"
    with open(json_path, 'w') as f:
        json.dump(metadata_list, f, indent=2)
    
    return metadata_list

print("✅ Processing functions defined")

## 🚀 Step 4: Process All BraTS Patients

This will take a while (~1-2 hours for 369 patients).

In [ ]:
# ============================================
# PROCESS ALL PATIENTS
# ============================================

# Check if BraTS data exists
if not BRATS_ROOT.exists():
    print(f"⚠️ BraTS data not found at {BRATS_ROOT}")
    print(f"\nPlease download BraTS 2020 and extract to:")
    print(f"  {BRATS_ROOT.parent}")
    print(f"\nSee: datasets_info/BraTS_Registration_Guide.md for instructions")
else:
    # Get all patient directories
    patient_dirs = sorted([d for d in BRATS_ROOT.iterdir() if d.is_dir()])
    print(f"Found {len(patient_dirs)} patients to process\n")
    
    # Process each patient
    all_metadata = []
    failed = []
    
    for patient_dir in tqdm(patient_dirs, desc="Processing patients"):
        try:
            metadata = process_patient(
                patient_dir,
                OUTPUT_DIR,
                survival_dict=survival_dict if 'survival_dict' in locals() else None,
                num_slices=10
            )
            if metadata:
                all_metadata.extend(metadata)
        except Exception as e:
            print(f"\n⚠️ Error processing {patient_dir.name}: {e}")
            failed.append(patient_dir.name)
    
    # Save combined metadata
    metadata_df = pd.DataFrame(all_metadata)
    metadata_df.to_csv(OUTPUT_DIR / 'all_metadata.csv', index=False)
    
    print(f"\n{'='*60}")
    print(f"✅ PROCESSING COMPLETE!")
    print(f"{'='*60}")
    print(f"Total images: {len(all_metadata)}")
    print(f"Failed patients: {len(failed)}")
    print(f"Output directory: {OUTPUT_DIR}")
    print(f"\nMetadata saved to: {OUTPUT_DIR / 'all_metadata.csv'}")
    print(f"\nYou can now train the conditional diffusion model!")
    
    if failed:
        print(f"\n⚠️ Failed patients: {failed}")

## 📊 Step 5: Explore the Processed Data

In [ ]:
# ============================================
# DATA EXPLORATION
# ============================================

# Load metadata
metadata_path = OUTPUT_DIR / 'all_metadata.csv'
if metadata_path.exists():
    df = pd.read_csv(metadata_path)
    
    print(f"Total images: {len(df)}")
    print(f"\nColumns: {list(df.columns)}")
    
    print(f"\nTumor grade distribution:")
    print(df['tumor_grade'].value_counts())
    
    print(f"\nBrain region distribution:")
    print(df['brain_region'].value_counts().head(10))
    
    print(f"\nTumor characteristics:")
    print(f"  With necrosis: {df['has_necrosis'].sum()}")
    print(f"  With edema: {df['has_edema'].sum()}")
    print(f"  With enhancement: {df['has_enhancement'].sum()}")
    
    print(f"\nAge statistics:")
    print(df['age'].describe())
    
    print(f"\nTumor volume statistics (mm³):")
    print(df['tumor_volume_mm3'].describe())
    
    # Sample
    print(f"\nSample data:")
    display(df.head(10))
else:
    print(f"⚠️ Metadata file not found. Run processing first.")

In [ ]:
# ============================================
# VISUALIZE PROCESSED SAMPLES
# ============================================

import random

if metadata_path.exists():
    # Sample 8 random images
    samples = df.sample(8)
    
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    axes = axes.flatten()
    
    for i, (_, row) in enumerate(samples.iterrows()):
        # Load image
        img_path = OUTPUT_DIR / row['patient_id'] / row['filename']
        img = Image.open(img_path)
        
        # Display
        axes[i].imshow(img, cmap='gray')
        axes[i].set_title(
            f"{row['brain_region']}\n"
            f"Grade: {row['tumor_grade']} | Age: {row['age']:.0f}\n"
            f"Volume: {row['tumor_volume_mm3']:.0f} mm³",
            fontsize=9
        )
        axes[i].axis('off')
    
    plt.suptitle('Sample Processed BraTS Images with Metadata', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print(f"⚠️ Process data first!")

# 🎯 Next Steps

Once preprocessing is complete, you'll have:

✅ **~3,690 PNG images** (369 patients × 10 slices each)  
✅ **Complete metadata** for each image (age, location, volume, grade)  
✅ **Ready to train** the diagnostic visualization model!

---

## Run Next Notebook:
**BraTS_Conditioned_Training.ipynb** - Train the full diagnostic diffusion model

This model will be able to:
```python
generate_diagnostic_scan(
    tumor_grade='HGG',
    brain_region='right_frontal_middle',
    tumor_volume=15000,
    age=56,
    has_necrosis=True,
    has_edema=True
)
```

And it will generate a brain MRI showing a glioblastoma in the right frontal lobe!